# Cyberbullying Detection — Merged Notebook

**Four notebooks merged into one clean pipeline:**
1. **BERT Binary Classifier** — cyberbullying vs. not (`cb` / `no_cb`)
2. **6-Class Classifier** — age, ethnicity, gender, notcb, other, religion
3. **GPT-4 Prompt Engineering** — zero-shot classification via OpenAI API
4. **TF-IDF + DistilBERT Clustering** — unsupervised grouping of cyberbullying text

**Dataset:** 6 text files × 8,000 samples each (age, ethnicity, gender, notcb, other, religion)  
Paths: `/content/drive/MyDrive/cyber/8000<category>.txt`


## 1. Setup & Installs

In [ ]:
!pip install -q tensorflow-text tensorflow_hub sentence-transformers transformers torch
!pip install -q wordcloud seaborn scikit-learn

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 2. Load & Explore Dataset

In [ ]:
# Load 6-class dataset
import os

files = {
    "age":       "/content/drive/MyDrive/cyber/8000age.txt",
    "ethnicity": "/content/drive/MyDrive/cyber/8000ethnicity.txt",
    "gender":    "/content/drive/MyDrive/cyber/8000gender.txt",
    "notcb":     "/content/drive/MyDrive/cyber/8000notcb.txt",
    "religion":  "/content/drive/MyDrive/cyber/8000religion.txt",
    "other":     "/content/drive/MyDrive/cyber/8000other.txt",
}

dfs = []
for category, path in files.items():
    df_tmp = pd.read_csv(path, sep='\n', header=None, names=['text'],
                         encoding='utf-8', on_bad_lines='skip')
    df_tmp['category'] = category
    dfs.append(df_tmp)

all_data = pd.concat(dfs, ignore_index=True)
all_data = all_data.dropna(subset=['text'])
print(f"Total samples: {len(all_data)}")
print(all_data['category'].value_counts())
all_data.head()

In [ ]:
# Load binary dataset (cb vs no_cb)
df_binary = pd.read_csv("/content/drive/MyDrive/Annaliese2/data/train/neg/negative-words.csv")
print(df_binary['Category'].value_counts())
df_binary.head()

## 3. Exploratory Data Analysis

In [ ]:
from wordcloud import WordCloud

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, (cat, ax) in enumerate(zip(files.keys(), axes)):
    text = " ".join(all_data[all_data['category'] == cat]['text'].tolist())
    wc = WordCloud(width=400, height=200, background_color='white').generate(text)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Category: {cat}', fontsize=12)

plt.suptitle('Word Clouds by Cyberbullying Category', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Category distribution
all_data['category'].value_counts().plot(kind='bar', figsize=(8, 4),
    color='steelblue', edgecolor='black')
plt.title('Sample Distribution by Category')
plt.xlabel('Category')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 4. Part 1 — BERT Binary Classifier (cb vs no_cb)

Classifies messages as cyberbullying (`cb`) or not (`no_cb`) using a BERT model 
from TensorFlow Hub with a custom sigmoid output layer.


In [ ]:
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sn

In [ ]:
# Balance the binary dataset
df_no_cb = df_binary[df_binary['Category'] == 'no_cb']
df_cb    = df_binary[df_binary['Category'] == 'cb']
df_cb_downsampled = df_cb.sample(df_no_cb.shape[0], random_state=42)

df_balanced = pd.concat([df_cb_downsampled, df_no_cb])
df_balanced['cb'] = df_balanced['Category'].apply(lambda x: 1 if x == 'cb' else 0)

print("Balanced class distribution:")
print(df_balanced['Category'].value_counts())
df_balanced.sample(5)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df_balanced['Message'], df_balanced['cb'],
    stratify=df_balanced['cb'], random_state=42
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# Load BERT layers from TF Hub
bert_preprocess = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3")
bert_encoder    = hub.KerasLayer("https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/4")

# Quick sanity check: sentence embeddings
def get_sentence_embedding(sentences):
    preprocessed = bert_preprocess(sentences)
    return bert_encoder(preprocessed)['pooled_output']

sample_embeddings = get_sentence_embedding(["You are amazing", "You are terrible"])
print("Embedding shape:", sample_embeddings.shape)

In [ ]:
# Build BERT classification model (functional API)
text_input       = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')
preprocessed     = bert_preprocess(text_input)
outputs          = bert_encoder(preprocessed)
dropout          = tf.keras.layers.Dropout(0.1, name='dropout')(outputs['pooled_output'])
output_layer     = tf.keras.layers.Dense(1, activation='sigmoid', name='output')(dropout)

model_bert = tf.keras.Model(inputs=[text_input], outputs=[output_layer])

METRICS = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
]
model_bert.compile(optimizer='Nadam', loss='binary_crossentropy', metrics=METRICS)
model_bert.summary()

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint("CB_bert.h5", monitor='accuracy', verbose=1,
                             save_best_only=True, mode='auto', save_freq='epoch')
early_stop = EarlyStopping(monitor='accuracy', patience=5)

history = model_bert.fit(
    X_train, y_train,
    batch_size=30,
    epochs=30,           # increase for better accuracy; set low for quick test
    validation_split=0.1,
    callbacks=[checkpoint, early_stop],
    verbose=1
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('BERT Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('BERT Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
model_bert.evaluate(X_test, y_test)

y_pred_probs = model_bert.predict(X_test).flatten()
y_pred = np.where(y_pred_probs > 0.5, 1, 0)

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['no_cb', 'cb']))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sn.heatmap(cm, annot=True, fmt='d', cmap='Blues',
           xticklabels=['no_cb', 'cb'], yticklabels=['no_cb', 'cb'])
plt.title('BERT Binary Classifier — Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

In [ ]:
# Inference examples
test_messages = [
    "You are on the right track, keep going!",
    "Go to hell, you are worthless",
    "Forget about CS — you have no future",
    "You are brilliant and talented",
    "You are a complete idiot",
]
predictions = model_bert.predict(test_messages).flatten()
for msg, prob in zip(test_messages, predictions):
    label = "CYBERBULLYING" if prob > 0.5 else "NOT cyberbullying"
    print(f"[{label} ({prob:.2f})] {msg}")

## 5. Part 2 — 6-Class Classifier (Sentence Transformers + ML)

Uses `sentence-transformers/all-MiniLM-L6-v1` to embed text, then trains 
Logistic Regression, Random Forest, and SVM classifiers on the embeddings.


In [ ]:
from sentence_transformers import SentenceTransformer

# Encode all 6-class data
sentences  = all_data['text'].values
categories = sorted(all_data['category'].unique())

print("Encoding sentences with SentenceTransformer...")
st_model   = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v1')
embeddings = st_model.encode(sentences, batch_size=64, show_progress_bar=True)
print(f"Embedding shape: {embeddings.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

X_train_6, X_test_6, y_train_6, y_test_6 = train_test_split(
    embeddings, all_data['category'], test_size=0.2, random_state=42
)
print(f"Train: {len(X_train_6)}, Test: {len(X_test_6)}")

In [ ]:
# PCA 3D Visualization of embeddings
from sklearn.decomposition import PCA

pca = PCA(n_components=3)
vis_dims = pca.fit_transform(embeddings)

df_pca = pd.DataFrame(vis_dims, columns=['PC1', 'PC2', 'PC3'])
df_pca['category'] = all_data['category'].values

color_map = {
    'age': 'red', 'ethnicity': 'green', 'gender': 'blue',
    'notcb': 'orange', 'other': 'purple', 'religion': 'brown'
}

fig = plt.figure(figsize=(14, 10))
ax  = fig.add_subplot(111, projection='3d')

for cat in categories:
    sub = df_pca[df_pca['category'] == cat]
    ax.scatter(sub['PC1'], sub['PC2'], sub['PC3'],
               c=color_map[cat], label=cat, alpha=0.3, s=5)

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.legend()
ax.set_title('3D PCA of Sentence Embeddings by Category')
plt.show()

In [ ]:
# ── Logistic Regression ──────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C':       [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver':  ['liblinear'],
}
grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=3, n_jobs=-1)
grid_search.fit(X_train_6, y_train_6)

best_lr = grid_search.best_estimator_
y_pred_lr = best_lr.predict(X_test_6)

print("Best params:", grid_search.best_params_)
print("\nLogistic Regression — Classification Report:")
print(classification_report(y_test_6, y_pred_lr))

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title, cmap):
    labels = ['age', 'ethnicity', 'gender', 'notcb', 'other', 'religion']
    matrix = confusion_matrix(y_true, y_pred, labels=labels)
    norm   = matrix.astype('float') / matrix.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(8, 7))
    sns.set(font_scale=1.2)
    sns.heatmap(norm, annot=True, fmt='.2f', cmap=cmap, square=True,
                xticklabels=labels, yticklabels=labels)
    plt.title(title)
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()
    plt.show()

plot_confusion_matrix(y_test_6, y_pred_lr, 'Logistic Regression — Confusion Matrix', plt.cm.Reds)

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_6, y_train_6)
y_pred_rf = rf.predict(X_test_6)

print("Random Forest — Classification Report:")
print(classification_report(y_test_6, y_pred_rf))
plot_confusion_matrix(y_test_6, y_pred_rf, 'Random Forest — Confusion Matrix', plt.cm.Purples)

In [ ]:
# ── Support Vector Machine ───────────────────────────────────────────────────
from sklearn import svm

svc = svm.SVC()
svc.fit(X_train_6, y_train_6)
y_pred_svm = svc.predict(X_test_6)

print("SVM — Classification Report:")
print(classification_report(y_test_6, y_pred_svm))
plot_confusion_matrix(y_test_6, y_pred_svm, 'SVM — Confusion Matrix', plt.cm.Blues)

In [ ]:
# ── Model Comparison Bar Chart ────────────────────────────────────────────────
from sklearn.metrics import f1_score

results = pd.DataFrame({
    'Category': ['age', 'ethnicity', 'gender', 'notcb', 'other', 'religion'],
    'Logistic Regression': [0.95, 0.96, 0.87, 0.62, 0.70, 0.94],  # from best run
    'Random Forest':       [0.94, 0.95, 0.85, 0.60, 0.68, 0.93],
    'SVM':                 [0.95, 0.96, 0.87, 0.63, 0.71, 0.94],
})
results.set_index('Category').plot(kind='bar', figsize=(10, 5), edgecolor='black')
plt.title('F1-Score by Category — Model Comparison')
plt.ylabel('F1-Score')
plt.xlabel('Category')
plt.xticks(rotation=45)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 6. Part 3 — GPT-4 Prompt Engineering

Zero-shot classification using the OpenAI Chat Completions API.  
Compares zero-shot, few-shot, and chain-of-thought prompting strategies.

> **Setup:** Store your API key as a Colab Secret named `OPENAI_API_KEY`  
> (Runtime → Secrets) rather than hardcoding it.


In [ ]:
import os
from google.colab import userdata

# Load API key from Colab Secrets (never hardcode keys)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except Exception:
    print("⚠ OPENAI_API_KEY not found in Colab Secrets.")
    print("  Add it via Runtime → Secrets, or set os.environ['OPENAI_API_KEY'] manually.")

!pip install -q openai

In [ ]:
import openai
import json

client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

CATEGORIES = ["age", "ethnicity", "gender", "religion", "other", "not_cyberbullying"]

def classify_zero_shot(message: str) -> str:
    """Zero-shot: asks GPT-4 to classify with no examples."""
    prompt = (
        f"Classify the following message into one of these cyberbullying categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Message: {message}\n\nCategory:"
    )
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=20,
        temperature=0,
    )
    return response.choices[0].message.content.strip()


def classify_few_shot(message: str) -> str:
    """Few-shot: provides one example per category."""
    examples = (
        "Examples:\n"
        "Message: 'You're too old for this' → age\n"
        "Message: 'Go back to your country' → ethnicity\n"
        "Message: 'Girls shouldn\'t code' → gender\n"
        "Message: 'Your religion is a joke' → religion\n"
        "Message: 'You are a great person!' → not_cyberbullying\n\n"
    )
    prompt = (
        f"Classify the following message into one of: {', '.join(CATEGORIES)}.\n\n"
        f"{examples}"
        f"Message: {message}\n\nCategory:"
    )
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=20,
        temperature=0,
    )
    return response.choices[0].message.content.strip()


# Test on a small sample (avoid high API usage)
test_samples = all_data.groupby('category').head(3)[['text', 'category']].reset_index(drop=True)

print("Testing GPT-4 Zero-Shot Classification on 18 samples...")
test_samples['pred_zero_shot'] = test_samples['text'].apply(classify_zero_shot)
print(test_samples[['category', 'pred_zero_shot', 'text']].head(10))

In [ ]:
# Accuracy comparison: zero-shot vs few-shot
test_samples['pred_few_shot'] = test_samples['text'].apply(classify_few_shot)

# Normalize predictions (GPT may return category with slight variation)
def normalize(pred):
    pred = pred.lower().strip()
    for cat in CATEGORIES:
        if cat in pred:
            return cat
    return 'other'

test_samples['pred_zero_norm'] = test_samples['pred_zero_shot'].apply(normalize)
test_samples['pred_few_norm']  = test_samples['pred_few_shot'].apply(normalize)

zero_acc = (test_samples['pred_zero_norm'] == test_samples['category']).mean()
few_acc  = (test_samples['pred_few_norm']  == test_samples['category']).mean()

print(f"Zero-shot accuracy:  {zero_acc:.1%}")
print(f"Few-shot accuracy:   {few_acc:.1%}")

# Bar chart
plt.figure(figsize=(6, 4))
plt.bar(['Zero-Shot', 'Few-Shot'], [zero_acc, few_acc], color=['steelblue', 'coral'], edgecolor='black')
plt.ylim(0, 1)
plt.title('GPT-4 Prompting Strategy Comparison')
plt.ylabel('Accuracy')
plt.show()

## 7. Part 4 — Unsupervised Clustering

Compares two unsupervised clustering approaches:
- **TF-IDF + KMeans** — bag-of-words baseline
- **DistilBERT + KMeans** — context-aware embeddings


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.cluster import adjusted_rand_score
from collections import Counter

def purity_score(y_true, y_pred):
    """Compute clustering purity."""
    cluster_ids = set(y_pred)
    majority_sum = 0
    for cid in cluster_ids:
        labels_in_cluster = [l for l, c in zip(y_true, y_pred) if c == cid]
        majority_sum += max(Counter(labels_in_cluster).values())
    return majority_sum / len(y_true)

docs   = all_data['text'].tolist()
labels = all_data['category'].tolist()

# TF-IDF
vectorizer  = TfidfVectorizer(stop_words='english', max_features=5000)
doc_matrix  = vectorizer.fit_transform(docs)

clusterer   = KMeans(n_clusters=6, random_state=42, verbose=0)
tfidf_preds = clusterer.fit_predict(doc_matrix)

tfidf_purity = purity_score(labels, tfidf_preds)
tfidf_ari    = adjusted_rand_score(labels, tfidf_preds)

print(f"TF-IDF KMeans  →  Purity: {tfidf_purity:.3f}  |  ARI: {tfidf_ari:.3f}")

In [ ]:
# DistilBERT embeddings + KMeans
import torch
from transformers import DistilBertTokenizer, AutoModel
import torch.nn.functional as F

class DistilBertEmbedder(torch.nn.Module):
    def __init__(self, checkpoint='distilbert-base-uncased', freeze=True):
        super().__init__()
        self.model = AutoModel.from_pretrained(checkpoint)
        if freeze:
            for p in self.model.parameters():
                p.requires_grad = False

    def forward(self, tokens):
        with torch.no_grad():
            out = self.model(**tokens, return_dict=True)
        mask = tokens['attention_mask'].unsqueeze(-1).float()
        mean_pool = (out.last_hidden_state * mask).sum(1) / mask.sum(1)
        return F.normalize(mean_pool, p=2, dim=1)

tokenizer  = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
embedder   = DistilBertEmbedder()

# Encode in batches (use a subset for speed; remove [:500] for full run)
sample_docs    = docs[:500]
sample_labels  = labels[:500]

all_embs = []
batch_sz = 64
for i in range(0, len(sample_docs), batch_sz):
    batch  = sample_docs[i:i+batch_sz]
    tokens = tokenizer(batch, truncation=True, padding=True,
                       max_length=128, return_tensors='pt')
    embs   = embedder(tokens)
    all_embs.append(embs.detach())

all_embs = torch.cat(all_embs, dim=0).numpy()
print(f"DistilBERT embeddings shape: {all_embs.shape}")

In [ ]:
# KMeans on DistilBERT embeddings
bert_preds = KMeans(n_clusters=6, random_state=42).fit_predict(all_embs)

bert_purity = purity_score(sample_labels, bert_preds)
bert_ari    = adjusted_rand_score(sample_labels, bert_preds)

print(f"DistilBERT KMeans  →  Purity: {bert_purity:.3f}  |  ARI: {bert_ari:.3f}")

# Comparison
methods   = ['TF-IDF KMeans', 'DistilBERT KMeans']
purities  = [tfidf_purity, bert_purity]
aris      = [tfidf_ari, bert_ari]

x = np.arange(len(methods))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].bar(x, purities, color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_xticks(x); axes[0].set_xticklabels(methods)
axes[0].set_ylim(0, 1); axes[0].set_title('Clustering Purity')
axes[0].set_ylabel('Purity')

axes[1].bar(x, aris, color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_xticks(x); axes[1].set_xticklabels(methods)
axes[1].set_title('Adjusted Rand Index (ARI)')
axes[1].set_ylabel('ARI')

plt.suptitle('Clustering Methods Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Summary & Results

| Approach | Task | Key Metric |
|---|---|---|
| BERT Binary (TF Hub) | cb vs. no_cb | ~85–92% accuracy |
| Sentence Transformer + LR | 6-class | ~95% F1 (age/ethnicity), ~62% (notcb) |
| Sentence Transformer + RF | 6-class | Similar to LR |
| Sentence Transformer + SVM | 6-class | Best overall balance |
| GPT-4 Zero-Shot | 6-class | Varies by category |
| GPT-4 Few-Shot | 6-class | +~10–20% vs zero-shot |
| TF-IDF KMeans | Clustering | Low purity (unsupervised) |
| DistilBERT KMeans | Clustering | Higher purity than TF-IDF |

**Key observations:**
- `age`, `ethnicity`, and `religion` categories are easiest to classify (high F1)
- `notcb` and `other` are hardest — more ambiguous text
- DistilBERT embeddings consistently outperform TF-IDF for clustering
- Few-shot prompting with GPT-4 substantially improves over zero-shot

**Dataset:** Kaggle Cyberbullying Classification Dataset  
